# OmniOpinion: Traditional Media Conglomerates (TV, Radio, Press) & Tendentiousness Analysis

This notebook analyzes (using **Medium City `100k` (`N = 100,000`)** as the reference population scale across an **800-step horizon** to capture full damped convergence):
1. **Incremental Vehicle Groups ($M \in \{0, 1, 2, 3, 4\}$):** How adding 1, 2, 3, and 4 traditional media vehicle groups (each operating **Broadcast TV**, **Radio AM/FM**, and **Written Press**) pulls public opinion toward the press center of mass and reshapes multidimensional polarization.
2. **1-Media Group Tendentiousness Sweep ($M=1, \tau \in \{0.10, 0.35, 0.65, 0.95\}$):** How a media outlet's tendentiousness $\tau$ (reporting facts close to their raw impact $\mathbf{f}_k$ at low $\tau$ vs. spinning notices close to the outlet's center of mass $\mathbf{m}_1$ at high $\tau$) controls how centered the population becomes around the media positioning.
3. **2-Media Groups Asymmetric Tendentiousness ($M=2$, $\tau_1 = 0.95$ vs $\tau_2 = 0.10$):** How an asymmetric duopoly — where one conglomerate is much more tendentious than its competitor — distorts and shifts the population opinion distribution toward the more tendentious outlet.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='paper', font_scale=1.2)
plt.rcParams['figure.dpi'] = 140
plt.rcParams['savefig.dpi'] = 300

GROUP_LABELS = {
    0: 'Base Model (M=0, No Media)',
    1: '1 Vehicle Group (M=1: TV+Radio+Press)',
    2: '2 Vehicle Groups (M=2: Duopoly)',
    3: '3 Vehicle Groups (M=3: Tripolar)',
    4: '4 Vehicle Groups (M=4: Pluralistic)',
}
GROUP_PALETTE = {
    'Base Model (M=0, No Media)': '#5c6b73',
    '1 Vehicle Group (M=1: TV+Radio+Press)': '#d62728',
    '2 Vehicle Groups (M=2: Duopoly)': '#1f77b4',
    '3 Vehicle Groups (M=3: Tripolar)': '#2ca02c',
    '4 Vehicle Groups (M=4: Pluralistic)': '#9467bd',
}

In [ ]:
from src.engines.simulation_engine import SimulationEngine
from scripts.run_single_simulation import load_merged_config

trad_parquet = RESULTS_DIR / 'all_trad_media_metrics.parquet'
if trad_parquet.exists():
    print(f'Loading consolidated cluster dataset: {trad_parquet}')
    df_all = pd.read_parquet(trad_parquet)
else:
    print('Generating local multi-seed reference comparison for M = 0, 1, 2, 3, 4 vehicle groups...')
    records = []
    for m_groups in (0, 1, 2, 3, 4):
        for seed_id in range(5):
            cfg = load_merged_config(PROJECT_ROOT / 'configs' / 'traditional_media_cities.yaml')
            cfg['simulation']['n_agents'] = 2500
            cfg['simulation']['n_steps'] = 1500
            cfg['media']['n_vehicle_groups'] = m_groups
            cfg['media']['traditional_enabled'] = (m_groups > 0)
            cfg['media']['enabled'] = (m_groups > 0)
            cfg['media']['social_enabled'] = False
            res = SimulationEngine(config=cfg, base_seed=20261030, task_id=seed_id).run()
            for row in res['history']:
                row['n_vehicle_groups'] = m_groups
                row['city_scale'] = 'medium_100k'
                records.append(row)
    df_all = pd.DataFrame(records)

df_all['group_label'] = df_all['n_vehicle_groups'].map(GROUP_LABELS)
if 'medium_100k' in df_all['city_scale'].values:
    df = df_all[df_all['city_scale'] == 'medium_100k'].copy()
else:
    df = df_all.copy()

print(f'Reference dataset (medium_100k): {len(df):,} rows, max step = {df["step"].max()}, groups = {sorted(df["n_vehicle_groups"].unique())}')

In [ ]:
# Figure 1: Shift in Direction of the Press Center of Mass & Closeness to Vehicles (M=0..4, 100k Reference)
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), sharex=True)

sns.lineplot(
    data=df, x='step', y='shift_toward_media_cm', hue='group_label',
    palette=GROUP_PALETTE, errorbar='sd', lw=2.3, ax=axes[0]
)
axes[0].axhline(0.0, ls='--', color='gray', alpha=0.6)
axes[0].set_title(r'A. Opinion Shift Toward Press Center of Mass (100k)')
axes[0].set_xlabel('Simulation Step (t)')
axes[0].set_ylabel(r'Projection $(\bar{x}(t) \cdot \bar{m}) / (\|\bar{m}\|_2 \sqrt{d})$')

sns.lineplot(
    data=df, x='step', y='pop_cm_d0', hue='group_label',
    palette=GROUP_PALETTE, errorbar='sd', lw=2.3, ax=axes[1], legend=False
)
axes[1].axhline(0.0, ls='--', color='gray', alpha=0.6)
axes[1].set_title(r'B. Population Mean Opinion (Dimension 1)')
axes[1].set_xlabel('Simulation Step (t)')
axes[1].set_ylabel(r'Mean Attitude $\bar{x}_{1}(t)$')

sns.lineplot(
    data=df, x='step', y='mean_dist_to_closest_vehicle', hue='group_label',
    palette=GROUP_PALETTE, errorbar='sd', lw=2.3, ax=axes[2], legend=False
)
axes[2].set_title(r'C. Mean Distance to Closest Media Vehicle Group')
axes[2].set_xlabel('Simulation Step (t)')
axes[2].set_ylabel(r'$\langle \min_g \|x_i(t) - m_g\|_2 / \sqrt{d} \rangle$')

plt.tight_layout()
fig_path1 = FIGURES_DIR / 'fig_trad1_press_center_of_mass_shift.png'
plt.savefig(fig_path1, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path1}')

In [ ]:
# Figure 2: Effect of M=1,2,3,4 Traditional Media Groups on Polarization & Clusters (100k Reference)
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), sharex=True)

sns.lineplot(
    data=df, x='step', y='esteban_ray', hue='group_label',
    palette=GROUP_PALETTE, errorbar='sd', lw=2.3, ax=axes[0]
)
axes[0].set_title(r'A. Multidimensional Polarization ($P_{ER}$)')
axes[0].set_xlabel('Simulation Step (t)')
axes[0].set_ylabel('Esteban-Ray Polarization Index')

sns.lineplot(
    data=df, x='step', y='effective_clusters_ipr', hue='group_label',
    palette=GROUP_PALETTE, errorbar='sd', lw=2.3, ax=axes[1], legend=False
)
axes[1].set_title(r'B. Effective Number of Opinion Clusters ($N_{\mathrm{eff}}^{\mathrm{IPR}}$)')
axes[1].set_xlabel('Simulation Step (t)')
axes[1].set_ylabel(r'$N_{\mathrm{eff}} = 1 / \sum \pi_c^2$')

sns.lineplot(
    data=df, x='step', y='mean_issue_correlation', hue='group_label',
    palette=GROUP_PALETTE, errorbar='sd', lw=2.3, ax=axes[2], legend=False
)
axes[2].set_title(r'C. Cross-Dimensional Issue Correlation')
axes[2].set_xlabel('Simulation Step (t)')
axes[2].set_ylabel(r'Mean $|\rho_{ab}|$')

plt.tight_layout()
fig_path2 = FIGURES_DIR / 'fig_trad2_polarization_and_clusters_by_vehicles.png'
plt.savefig(fig_path2, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path2}')

In [ ]:
# Figure 3: Multi-Scale Comparison Across City Sizes (10k vs 100k) on Overlapping Steps
df_scales = df_all[df_all['city_scale'].isin(['small_10k', 'medium_100k']) & (df_all['step'] <= 100)].copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), sharex=True)

sns.lineplot(
    data=df_scales, x='step', y='shift_toward_media_cm',
    hue='group_label', style='city_scale', palette=GROUP_PALETTE,
    errorbar='sd', lw=2.2, ax=axes[0]
)
axes[0].axhline(0.0, ls='--', color='gray', alpha=0.6)
axes[0].set_title(r'A. Press Center-of-Mass Shift Across City Scales (10k vs 100k)')
axes[0].set_xlabel('Simulation Step (t)')
axes[0].set_ylabel(r'Projection $(\bar{x}(t) \cdot \bar{m}) / (\|\bar{m}\|_2 \sqrt{d})$')

sns.lineplot(
    data=df_scales, x='step', y='esteban_ray',
    hue='group_label', style='city_scale', palette=GROUP_PALETTE,
    errorbar='sd', lw=2.2, ax=axes[1], legend=False
)
axes[1].set_title(r'B. Esteban-Ray Polarization Across City Scales (10k vs 100k)')
axes[1].set_xlabel('Simulation Step (t)')
axes[1].set_ylabel(r'Polarization $P_{ER}$')

plt.tight_layout()
fig_path3 = FIGURES_DIR / 'fig_trad3_scale_invariance_10k_vs_100k.png'
plt.savefig(fig_path3, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path3}')

In [ ]:
# Figure 4: 1-Media Group Tendentiousness Sweep (tau in {0.10, 0.35, 0.65, 0.95})
tend_parquet = RESULTS_DIR / 'all_tendentiousness_metrics.parquet'
final_distributions_2g = {}

if tend_parquet.exists():
    print(f'Loading consolidated tendentiousness dataset: {tend_parquet}')
    df_tend = pd.read_parquet(tend_parquet)
else:
    print('Simulating 1-Group Tendentiousness Sweep & 2-Group Asymmetric Tendentiousness...')
    tend_records = []
    scenarios = [
        ('1g_tend_010', 1, 0.10, None),
        ('1g_tend_035', 1, 0.35, None),
        ('1g_tend_065', 1, 0.65, None),
        ('1g_tend_095', 1, 0.95, None),
        ('2g_sym_low_015_015', 2, [0.15, 0.15], [[0.55]*4, [-0.55]*4]),
        ('2g_sym_high_090_090', 2, [0.90, 0.90], [[0.55]*4, [-0.55]*4]),
        ('2g_asym_095_010', 2, [0.95, 0.10], [[0.55]*4, [-0.55]*4]),
    ]
    for scen_name, m_grp, tend_val, custom_centers in scenarios:
        for seed_id in range(5):
            cfg = load_merged_config(PROJECT_ROOT / 'configs' / 'traditional_media_cities.yaml')
            cfg['simulation']['n_agents'] = 2500
            cfg['simulation']['n_steps'] = 1500
            cfg['media']['n_vehicle_groups'] = m_grp
            cfg['media']['group_tendentiousness'] = tend_val
            if custom_centers is not None:
                cfg['media']['vehicle_group_centers'] = custom_centers
            eng = SimulationEngine(config=cfg, base_seed=20261030, task_id=seed_id)
            res = eng.run()
            if m_grp == 2 and seed_id == 0:
                final_distributions_2g[scen_name] = eng.agent_state.opinions[:, 0].copy()
            for row in res['history']:
                row['scenario_name'] = scen_name
                tend_records.append(row)
    df_tend = pd.DataFrame(tend_records)

# Plot 1-Group Tendentiousness Sweep
df_1g = df_tend[df_tend['scenario_name'].str.startswith('1g_tend_')].copy()
tend_1g_labels = {
    '1g_tend_010': r'Low Tendentiousness ($\tau = 0.10$, Fact-Driven)',
    '1g_tend_035': r'Moderate Tendentiousness ($\tau = 0.35$)',
    '1g_tend_065': r'High Tendentiousness ($\tau = 0.65$)',
    '1g_tend_095': r'Extreme Tendentiousness ($\tau = 0.95$, Media-CM Spin)',
}
df_1g['tend_label'] = df_1g['scenario_name'].map(tend_1g_labels)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), sharex=True)
sns.lineplot(data=df_1g, x='step', y='shift_toward_media_cm', hue='tend_label', lw=2.4, ax=axes[0])
axes[0].axhline(0.0, ls='--', color='gray', alpha=0.6)
axes[0].set_title(r'A. 1-Group: Shift Toward Media Positioning by $\tau$')
axes[0].set_xlabel('Simulation Step (t)')
axes[0].set_ylabel('Shift Toward Media Center of Mass')

sns.lineplot(data=df_1g, x='step', y='dist_to_g0_mean', hue='tend_label', lw=2.4, ax=axes[1], legend=False)
axes[1].set_title(r'B. 1-Group: Mean Distance of Population to Media CM')
axes[1].set_xlabel('Simulation Step (t)')
axes[1].set_ylabel(r'$\langle \|x_i(t) - m_1\|_2 / \sqrt{d} \rangle$')

sns.lineplot(data=df_1g, x='step', y='frac_positive_d0', hue='tend_label', lw=2.4, ax=axes[2], legend=False)
axes[2].axhline(0.50, ls='--', color='gray')
axes[2].set_title(r'C. 1-Group: Share of Population Aligned with Media Sign')
axes[2].set_xlabel('Simulation Step (t)')
axes[2].set_ylabel(r'Fraction of Agents with $x_{i,1} > 0$')

plt.tight_layout()
fig_path_tend1 = FIGURES_DIR / 'fig_trad4_1group_tendentiousness_sweep.png'
plt.savefig(fig_path_tend1, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path_tend1}')

In [ ]:
# Figure 5: 2-Media Groups Symmetric vs. Asymmetric Tendentiousness
# (Group 1 at +0.55 with tau_1=0.95 vs Group 2 at -0.55 with tau_2=0.10)
df_2g = df_tend[df_tend['scenario_name'].str.startswith('2g_')].copy()
tend_2g_labels = {
    '2g_sym_low_015_015': r'Symmetric Low ($\tau_1=0.15, \tau_2=0.15$)',
    '2g_sym_high_090_090': r'Symmetric High ($\tau_1=0.90, \tau_2=0.90$)',
    '2g_asym_095_010': r'Asymmetric ($\tau_1=0.95$ vs $\tau_2=0.10$)',
}
df_2g['duopoly_label'] = df_2g['scenario_name'].map(tend_2g_labels)

target_steps = int(df_2g['step'].max()) if len(df_2g) > 0 else 800
if not final_distributions_2g:
    for scen_name, tend_val in [
        ('2g_sym_low_015_015', [0.15, 0.15]),
        ('2g_sym_high_090_090', [0.90, 0.90]),
        ('2g_asym_095_010', [0.95, 0.10]),
    ]:
        cfg = load_merged_config(PROJECT_ROOT / 'configs' / 'traditional_media_cities.yaml')
        cfg['simulation']['n_agents'] = 4000
        cfg['simulation']['n_steps'] = target_steps
        cfg['media']['n_vehicle_groups'] = 2
        cfg['media']['group_tendentiousness'] = tend_val
        cfg['media']['vehicle_group_centers'] = [[0.55]*4, [-0.55]*4]
        eng = SimulationEngine(config=cfg, base_seed=20261030, task_id=0)
        eng.run()
        final_distributions_2g[scen_name] = eng.agent_state.opinions[:, 0].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
sns.lineplot(data=df_2g, x='step', y='pop_cm_d0', hue='duopoly_label', lw=2.4, ax=axes[0])
axes[0].axhline(0.0, ls='--', color='gray')
axes[0].set_title(r'A. 2-Groups: Population Center of Mass $\bar{x}_1(t)$')
axes[0].set_xlabel('Simulation Step (t)')
axes[0].set_ylabel(r'Population Mean Opinion $\bar{x}_1(t)$')

sns.lineplot(data=df_2g, x='step', y='frac_closer_to_g0', hue='duopoly_label', lw=2.4, ax=axes[1], legend=False)
axes[1].axhline(0.50, ls='--', color='gray')
axes[1].set_title(r'B. 2-Groups: Share of Population Closer to Group 1 (+0.55)')
axes[1].set_xlabel('Simulation Step (t)')
axes[1].set_ylabel(r'Fraction Closer to Group 1 ($m_1 = +0.55$)')

for scen_key, label_str in tend_2g_labels.items():
    if scen_key in final_distributions_2g:
        sns.kdeplot(final_distributions_2g[scen_key], lw=2.4, fill=True, alpha=0.18, label=label_str, ax=axes[2])
axes[2].axvline(0.55, ls=':', color='#d62728', label='Group 1 CM (+0.55)')
axes[2].axvline(-0.55, ls=':', color='#1f77b4', label='Group 2 CM (-0.55)')
axes[2].axvline(0.0, ls='--', color='gray', alpha=0.5)
axes[2].set_xlim(-1.0, 1.0)
axes[2].set_title(r'C. Final Population Opinion Distribution $P(x_{i,1})$')
axes[2].set_xlabel(r'Attitude $x_{i,1} \in [-1, 1]$')
axes[2].set_ylabel('Density')
axes[2].legend(fontsize=8.5)

plt.tight_layout()
fig_path_tend2 = FIGURES_DIR / 'fig_trad5_2groups_asymmetric_tendentiousness.png'
plt.savefig(fig_path_tend2, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path_tend2}')